# Práctica 4: Modelo binomial

## Ejercicios

El precio actual de una acción es de $50$ euros, y se sabe que dentro de 2 meses el precio se moverá a $53$ ó a $48$ euros.La tasa de interés con composición continua es del $6\%$ anual. Utiliza el modelo binomial para valorar una opción de compra con vencimiento $T$ dos meses y strike $K=50$. ¿Cuál es el precio de una opción de **venta** europea con los mismos parámetros? Calcula las carteras que replican ambos instrumentos.

In [ ]:
import numpy as np
# definimos los parámetros
S0 = 50
r = 0.06
K = 50
Su = 53
Sd = 48
T = 2/12 # fracción de año correspondiente a los dos meses
BT = np.exp(r*T)

## Hacemos como ejemplo el caso de una opción de compra

Definimos la probabilidad neutral al riesgo por medio de:
$$ q = \frac{S_0B_T - S_d}{S_u-S_d}$$
y calculamos el precio inicial de la opción Call usando:
$$ C_0 = \frac{1}{B_T}(qC_u + (1-q)C_d) $$ donde
$$ C_i = \max(S_i - K, 0)\quad i=u,d $$

## Pay-off de la opción

In [ ]:
def Call(S,K):
    return np.maximum(S-K,0)

q  = (S0*BT-Sd)/(Su-Sd)
# precio
C0 = (q*Call(Su,K)+(1-q)*Call(Sd,K))/BT

In [ ]:
C0

## Cartera de replicación

Calculamos ahora la cartera que replica a la opción Call:
$$\psi_{0}=\frac{C_u-C_d}{S_u-S_d}\quad\phi_{0}=\frac{S_u C_d - S_d C_u}{B_{T}\left(S_u-S_d\right)}\equiv C_0 - \psi_0 S_0$$

In [ ]:
Cu,Cd = Call(Su,K),Call(Sd,K)
psi = (Cu-Cd)/(Su-Sd)
phi = (Su*Cd - Sd*Cu)/(BT*(Su-Sd))

In [ ]:
psi

Comprobamos ahora que la cartera replica los valores de la opción:

In [ ]:
def V(S):
    return psi*S+phi*BT
V(Su),V(Sd)

## Modelo de Cox-Ross-Rubinstein en varios pasos

La forma más simple de implementar el modelo en varios pasos (aunque no la más eficiente) es mediante una función que devuelva los valores de $S$ en el paso $n=0,1,2\dots$ y en el "estado" $k$, donde $k$ indica el número de veces que el precio "sube", donde $k\leq n$. Consideramos un ejemplo.

In [ ]:
S0 = 80.0
K = 80
u,d = 0.2,-0.15 # factores del modelo
r = 0.06 # tipo de interés simple para el intervalo
dt = 1/4 # tamaño del intervalo temporal
N = 8 # número de pasos temporales
b = r*dt # le quito el 1 para no sumar y restar en la fórmula de q
B = 1 + b # factor de crecimiento en el intervalo
q = (b-d)/(u-d) # medida neutral al riesgo
# función que define el árbol de precios en términos de k= nro de pasos que crece, de un total de n
def S(k,n):
    return S0 * (1+u)**k * (1+d)**(n-k)

Usando los valores en el instante final, podemos definir el pago del instrumento:

In [ ]:
payoff = [Call(S(k,N),K) for k in np.arange(N+1)]

In [ ]:
payoff

In [ ]:
q

Guardamos los precios en un array:

In [ ]:
C = np.zeros((N+1,N+1))
C[:,-1] = payoff # incorporamos el pay-off a la última columna
# completamos a la matriz con los precios
for n in np.arange(N,0,-1):
    C[0:n,n-1] = (q*C[1:(n+1),n]+(1-q)*C[0:n,n])/B

In [ ]:
C[0,0] # precio de la opción Call con los parámetros dados

# Práctica
Una acción cotiza hoy a 100 euros. El tipo de interés bancario para todos los intervalos es de $5\%$ con composición continua (nótese que, para definir `b` hace falta hacer el cambio entre tipo continuo y simple). La evolución del precio de la acción puede modelarse con un modelo CRR con parámetros $u=0.1$ y $d=-0.1$. Los cambios en el precio se dan cada 1 mes ($\tau=1/12$). ¿Cuáles son los precios de las correspondientes opciones de compra y venta con vencimiento en 1 año si el strike $K$ es igual a 100?

Calcula los coeficientes $\phi,\psi$ de la cartera de cobertura para cada instante y cada estado. Comprueba que la replicación es exacta.

**Respuesta**: precio de la Call 16.186830758274404 euros, Put: 11.30977320834581 euros.
Valor de $\psi$ en el primer paso temporal: 0.6197604779335787